# Baseline: Gemma 3 clinical SOAP summaries (before fine-tuning)

This notebook generates summaries with the **original** `google/gemma-3-270m-it`, then uses DeepEval G-Eval to score factual faithfulness and coverage on 50 randomly selected validation examples. It does not train a model. Scores are approximate LLM judgments, not clinical validation.

**Before starting:** Open in Colab, select a GPU runtime, accept the [Gemma license](https://huggingface.co/google/gemma-3-270m-it), and create a Hugging Face access token. A Gemini API key with appropriate quota or billing is needed only for the separate judging step. Use Colab Secrets or environment variables, never paste keys into this notebook. For Colab Secrets, add `HF_TOKEN` and `GOOGLE_API_KEY` and enable notebook access.

**Cost and data:** Model/dataset downloads and GPU runtime may incur provider charges. The optional judge sends clinical notes, references and generated summaries to Google Gemini and may incur API charges: 50 cases × 2 metrics means at least 100 judge requests, with possible retries. Cost depends on token counts and current [Gemini API pricing](https://ai.google.dev/gemini-api/docs/pricing); inspect the selection and pricing first. Dataset sources are described as de-identified, but treat all saved and transmitted text as sensitive. Avoid Confident AI login for this demo. DeepEval telemetry is disabled below.

**Persistence:** Colab's local disk is temporary. Download the `baseline_evaluation` folder or set `OUTPUT_DIR` to a mounted Drive folder. Keep `selection.json` unchanged for a later fine-tuned comparison; it stores the exact inputs and references, indices, hashes and dataset revision. The saved JSON files contain clinical text.


In [ ]:
%pip install -q "transformers>=4.55,<5" "datasets>=3,<5" accelerate "deepeval>=4.3.3,<5" google-genai pandas


In [ ]:
import os, json, hashlib, random, importlib.metadata
from pathlib import Path
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"
os.environ["DEEPEVAL_EVAL_MODE"] = "llm"
# Colab Secrets are optional if environment variables already exist.
try:
    from google.colab import userdata
    for key in ("HF_TOKEN", "GOOGLE_API_KEY"):
        if not os.environ.get(key):
            try:
                value = userdata.get(key)
                if value:
                    os.environ[key] = value
            except Exception:
                pass
except ImportError:
    pass

DATASET_ID = "AmareshHebbar/clinical-summarizer-sft"
MODEL_ID = "google/gemma-3-270m-it"
SEED, N_CASES, MAX_NEW_TOKENS = 42, 50, 512
JUDGE_MODEL = "gemini-2.5-flash"  # Gemini API judge, separate from Gemma
OUTPUT_DIR = Path("baseline_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("CUDA available:", torch.cuda.is_available())
print("Output directory:", OUTPUT_DIR.resolve())


## 1. Freeze the validation sample

The first run samples without replacement using seed 42, resolves the dataset's current commit, and saves the full 50 examples. Subsequent runs load the existing selection rather than resampling. Save this file for your future comparison. Delete or rename the output folder only if you deliberately want a new experiment.


In [ ]:
selection_path = OUTPUT_DIR / "selection.json"
if selection_path.exists():
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    assert selection["dataset"] == DATASET_ID and len(selection["cases"]) == N_CASES
    print("Reusing saved selection")
else:
    revision = HfApi().dataset_info(DATASET_ID, token=os.environ.get("HF_TOKEN")).sha
    validation = load_dataset(DATASET_ID, revision=revision, split="validation")
    indices = random.Random(SEED).sample(range(len(validation)), N_CASES)
    cases = []
    for index in indices:
        messages = validation[index]["messages"]
        by_role = {m["role"]: m["content"] for m in messages}
        assert all(by_role.get(role) for role in ("system", "user", "assistant")), index
        raw = json.dumps(messages, ensure_ascii=False, sort_keys=True).encode("utf-8")
        cases.append({"index": index, "sha256": hashlib.sha256(raw).hexdigest(),
                      "system": by_role["system"], "note": by_role["user"],
                      "reference": by_role["assistant"]})
    selection = {"dataset": DATASET_ID, "revision": revision, "split": "validation",
                 "seed": SEED, "cases": cases}
    selection_path.write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding="utf-8")
print("Dataset revision:", selection["revision"])
print("Selected validation row indices:", [c["index"] for c in selection["cases"]])
print("Saved:", selection_path)


## 2. Generate baseline summaries

Gemma receives each dataset system instruction and note, combined into a user message, as in the training notebook. Greedy decoding makes generation repeatable for a fixed software/hardware setup. The reference summary is never sent to Gemma. Results are checkpointed after every case; a stopped run can be resumed. Start with a fresh output directory if you change generation settings.


In [ ]:
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Set HF_TOKEN after accepting the Gemma license, then rerun this cell.")
model_revision = HfApi().model_info(MODEL_ID, token=os.environ["HF_TOKEN"]).sha
config = {"model": MODEL_ID, "model_revision": model_revision,
          "dataset_revision": selection["revision"], "seed": SEED,
          "max_new_tokens": MAX_NEW_TOKENS, "do_sample": False,
          "transformers": importlib.metadata.version("transformers"),
          "torch": torch.__version__}
config_path = OUTPUT_DIR / "generation_config.json"
if config_path.exists():
    assert json.loads(config_path.read_text(encoding="utf-8")) == config, "Config changed; use a new output directory."
else:
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_revision, token=os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=model_revision, token=os.environ["HF_TOKEN"],
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to("cuda" if torch.cuda.is_available() else "cpu").eval()
outputs_path = OUTPUT_DIR / "generations.json"
outputs = json.loads(outputs_path.read_text(encoding="utf-8")) if outputs_path.exists() else {}
for case in selection["cases"]:
    key = str(case["index"])
    if key in outputs:
        continue
    prompt = case["system"] + "\n\n" + case["note"]
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    outputs[key] = tokenizer.decode(generated[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    outputs_path.write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Generated {len(outputs)}/{N_CASES}: row {key}")
print("Saved:", outputs_path)


## 3. Local SOAP structure and word overlap (no judge)

These checks run locally after generation. The parser recognizes S/O/A/P or full section names, with common colon, dash, and Markdown heading styles, including inline headings. It retains repeated sections, text before the first heading, and original text. Missing, empty, repeated, and out-of-order flags describe the parsed structure; headings inside prose or unusual formatting may be misread. Inspect raw text alongside flags.

Reference word recall is the fraction of unique reference words also found in the generated summary; precision is the fraction of unique generated words found in the reference. These are simple case-insensitive word overlaps, not measures of factual correctness, clinical coverage, or semantic equivalence. Negation, paraphrases, and different word meanings can make overlap misleading. No judge, API key, or external call is needed for this step.


In [ ]:
import re
from collections import Counter

SOAP_NAMES = ("subjective", "objective", "assessment", "plan")
# shortcut: heading-like text in prose can be parsed as a section; inspect raw text or use a domain parser if needed.
HEADING = re.compile(
    r"(?<!\w)(?:\*\*|__)?(?P<label>(?i:subjective|objective|assessment|plan)|[SOAP])"
    r"(?:\*\*|__)?\s*(?::|[-–—])\s*(?:\*\*|__)?"
    r"|(?m:^[ \t]*(?:#{1,6}[ \t]*)?(?:\*\*|__)?(?P<standalone>(?i:subjective|objective|assessment|plan)|[SOAP])(?:\*\*|__)?[ \t]*$)"
)

def parse_soap(text):
    """Keep every recognized occurrence, its content, and structural flags."""
    text = text or ""
    matches = list(HEADING.finditer(text))
    aliases = dict(zip("soap", SOAP_NAMES))
    sections = []
    for i, match in enumerate(matches):
        label = (match.group("label") or match.group("standalone")).lower()
        name = aliases.get(label, label)
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[match.end():end].strip()
        sections.append({"name": name, "heading": match.group().strip(), "text": content})
    counts = Counter(s["name"] for s in sections)
    order = [SOAP_NAMES.index(s["name"]) for s in sections]
    return {"raw_text": text, "preamble": text[:matches[0].start()].strip() if matches else text.strip(),
            "sections": sections, "missing": [name for name in SOAP_NAMES if not counts[name]],
            "repeated": [name for name in SOAP_NAMES if counts[name] > 1],
            "empty": [s["name"] for s in sections if not s["text"]],
            "out_of_order": order != sorted(order)}

def word_overlap(generated, reference):
    words = lambda text: set(re.findall(r"\b\w+\b", text.casefold()))
    actual, expected = words(generated), words(reference)
    common = len(actual & expected)
    return {"reference_word_recall": common / len(expected) if expected else None,
            "generated_word_precision": common / len(actual) if actual else None}

# Small examples make the parser's behavior visible without loading a model.
check = parse_soap("Intro\n**S:** pain\nO: normal\nA:\nP: rest\nP: follow up")
assert check["preamble"] == "Intro" and check["repeated"] == ["plan"]
assert check["empty"] == ["assessment"] and not check["missing"] and not check["out_of_order"]
assert parse_soap("Plan: rest\nSubjective: pain")["out_of_order"]
assert parse_soap("no headings")["missing"] == list(SOAP_NAMES)
assert word_overlap("no fever", "fever")['reference_word_recall'] == 1.0  # Not factual agreement!
print("Parser self-checks passed.")


In [ ]:
outputs_path = OUTPUT_DIR / "generations.json"
outputs = json.loads(outputs_path.read_text(encoding="utf-8")) if outputs_path.exists() else {}
local_results = []
for case in selection["cases"]:
    key = str(case["index"])
    if key not in outputs:
        continue  # Missing generations are not counted as empty model outputs.
    local_results.append({"index": case["index"], "generated": parse_soap(outputs[key]),
                          "reference": parse_soap(case["reference"]),
                          "word_overlap": word_overlap(outputs[key], case["reference"])})
local_path = OUTPUT_DIR / "local_checks.json"
local_path.write_text(json.dumps(local_results, ensure_ascii=False, indent=2), encoding="utf-8")
local_summary = {"generated_cases_checked": len(local_results)}
for side in ("generated", "reference"):
    local_summary[side] = {flag + "_cases": sum(bool(row[side][flag]) for row in local_results)
                           for flag in ("missing", "repeated", "empty", "out_of_order")}
for name in ("reference_word_recall", "generated_word_precision"):
    values = [row["word_overlap"][name] for row in local_results if row["word_overlap"][name] is not None]
    local_summary[name] = {"count": len(values), "mean": sum(values) / len(values) if values else None}
(OUTPUT_DIR / "local_aggregate.json").write_text(json.dumps(local_summary, indent=2), encoding="utf-8")
print(json.dumps(local_summary, indent=2))
print("Per-case details:", local_path)


## 4. Judge with DeepEval (optional paid API calls)

Set `RUN_PAID_JUDGING = True` only after reviewing the cost and data transfer above. The Gemini Developer API judge is separate from the baseline model; it uses `GOOGLE_API_KEY`, and does not require an OpenAI key. The notebook does not configure a custom OpenAI-compatible base URL. Other providers require modifying the judge integration; see [DeepEval model integrations](https://deepeval.com/integrations). Each G-Eval metric scores 0–1 and provides a reason: **faithfulness** checks the source note for unsupported or contradicted claims; **coverage** checks whether important source and reference facts appear in the generated summary. References can themselves contain errors, so the source note is authoritative. These scores are noisy and are not proof of medical accuracy. Each completed score is saved immediately; rerun to resume after an interruption.


In [ ]:
RUN_PAID_JUDGING = False  # Change to True to authorize API calls for this run.
if not RUN_PAID_JUDGING:
    print("Judging skipped. Review costs and set RUN_PAID_JUDGING = True to evaluate.")
else:
    if not os.environ.get("GOOGLE_API_KEY"):
        raise RuntimeError("Set GOOGLE_API_KEY in Colab Secrets or the environment.")
    from deepeval.metrics import GEval
    from deepeval.models import GeminiModel
    from deepeval.test_case import LLMTestCase, SingleTurnParams

    assert len(outputs) == N_CASES, "Complete generation first."
    judge_config = {"provider": "Gemini Developer API", "model": JUDGE_MODEL, "deepeval": importlib.metadata.version("deepeval"),
                    "selection_sha256": hashlib.sha256(selection_path.read_bytes()).hexdigest(),
                    "generations_sha256": hashlib.sha256(outputs_path.read_bytes()).hexdigest()}
    judge_config_path = OUTPUT_DIR / "judge_config.json"
    if judge_config_path.exists():
        assert json.loads(judge_config_path.read_text(encoding="utf-8")) == judge_config, "Judge or inputs changed; use a new output directory."
    else:
        judge_config_path.write_text(json.dumps(judge_config, indent=2), encoding="utf-8")
    judge = GeminiModel(model=JUDGE_MODEL, api_key=os.environ["GOOGLE_API_KEY"], use_vertexai=False, temperature=0)
    metrics = {
        "faithfulness": GEval(name="Factual faithfulness", model=judge, async_mode=False,
            evaluation_steps=["Compare every claim in the actual output with the source note in input.",
                              "Penalize unsupported, invented, or contradicted clinical facts, especially diagnoses, medications and quantities."],
            evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT]),
        "coverage": GEval(name="Clinical coverage", model=judge, async_mode=False,
            evaluation_steps=["Identify important clinical facts in the source note in input, using expected output as a guide.",
                              "Check whether actual output preserves these facts; penalize important omissions. The source note is authoritative if the reference conflicts."],
            evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT]),
    }
    scores_path = OUTPUT_DIR / "scores.json"
    scores = json.loads(scores_path.read_text(encoding="utf-8")) if scores_path.exists() else {}
    for case in selection["cases"]:
        key = str(case["index"])
        scores.setdefault(key, {})
        test_case = LLMTestCase(input=case["note"], actual_output=outputs[key], expected_output=case["reference"])
        for name, metric in metrics.items():
            if name in scores[key]:
                continue
            metric.measure(test_case)
            scores[key][name] = {"score": metric.score, "reason": metric.reason}
            scores_path.write_text(json.dumps(scores, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"Row {key}, {name}: {metric.score:.3f}")
    print("Saved:", scores_path)


## 5. Inspect and export results

This cell reports available scores and their counts, including partial runs. `per_case.csv` includes the note, reference, output, scores and judge explanations. Compare later models on the same saved selection, prompt, generation settings and judge configuration; the judge may still vary between runs.


In [ ]:
outputs = json.loads((OUTPUT_DIR / "generations.json").read_text(encoding="utf-8")) if (OUTPUT_DIR / "generations.json").exists() else {}
scores = json.loads((OUTPUT_DIR / "scores.json").read_text(encoding="utf-8")) if (OUTPUT_DIR / "scores.json").exists() else {}
rows = []
for case in selection["cases"]:
    key = str(case["index"])
    row = {"index": case["index"], "sha256": case["sha256"], "note": case["note"],
           "reference": case["reference"], "generated_soap": outputs.get(key)}
    for name in ("faithfulness", "coverage"):
        result = scores.get(key, {}).get(name, {})
        row[name + "_score"] = result.get("score")
        row[name + "_reason"] = result.get("reason")
    rows.append(row)
results = pd.DataFrame(rows)
results.to_csv(OUTPUT_DIR / "per_case.csv", index=False)
aggregate = {"selected_cases": len(results), "generated_cases": int(results.generated_soap.notna().sum())}
for name in ("faithfulness", "coverage"):
    values = pd.to_numeric(results[name + "_score"], errors="coerce").dropna()
    aggregate[name] = {"count": int(values.count()), "mean": float(values.mean()) if len(values) else None}
(OUTPUT_DIR / "aggregate.json").write_text(json.dumps(aggregate, indent=2), encoding="utf-8")
print(json.dumps(aggregate, indent=2))
display(results[["index", "generated_soap", "faithfulness_score", "coverage_score"]].head())
